<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
using System;
using System.Collections.Generic;
using System.Linq;

// Делегаты для событий
public delegate void PaymentProcessedEventHandler(object sender, PaymentEventArgs e);
public delegate void PaymentFailedEventHandler(object sender, PaymentEventArgs e);
public delegate void PaymentStatusChangedEventHandler(object sender, StatusChangedEventArgs e);

// Аргументы событий
public class PaymentEventArgs : EventArgs
{
    public decimal Amount { get; set; }
    public string Currency { get; set; }
    public string TransactionId { get; set; }
    public DateTime Timestamp { get; set; }
    
    public PaymentEventArgs(decimal amount, string currency, string transactionId)
    {
        Amount = amount;
        Currency = currency;
        TransactionId = transactionId;
        Timestamp = DateTime.Now;
    }
}

public class StatusChangedEventArgs : EventArgs
{
    public string OldStatus { get; set; }
    public string NewStatus { get; set; }
    public string MethodName { get; set; }
    
    public StatusChangedEventArgs(string oldStatus, string newStatus, string methodName)
    {
        OldStatus = oldStatus;
        NewStatus = newStatus;
        MethodName = methodName;
    }
}

// Менеджер для работы с коллекциями платежей
public class PaymentManager
{
    // Коллекции для хранения платежей
    private Dictionary<int, PaymentMethod> _paymentsDictionary;
    private List<PaymentMethod> _paymentsList;
    private HashSet<string> _paymentNames;
    private Queue<PaymentMethod> _pendingPayments;

    public PaymentManager()
    {
        _paymentsDictionary = new Dictionary<int, PaymentMethod>();
        _paymentsList = new List<PaymentMethod>();
        _paymentNames = new HashSet<string>();
        _pendingPayments = new Queue<PaymentMethod>();
    }

    // Добавление платежа в коллекции
    public void AddPayment(PaymentMethod payment)
    {
        _paymentsDictionary[payment.PaymentMethodId] = payment;
        _paymentsList.Add(payment);
        _paymentNames.Add(payment.MethodName);
        _pendingPayments.Enqueue(payment);
        
        Console.WriteLine($"Платеж добавлен в менеджер: {payment.MethodName}");
    }

    // Поиск по ID
    public PaymentMethod FindPaymentById(int id)
    {
        return _paymentsDictionary.ContainsKey(id) ? _paymentsDictionary[id] : null;
    }

    // Фильтрация по минимальной сумме
    public List<PaymentMethod> GetPaymentsAboveAmount(decimal amount)
    {
        return _paymentsList.Where(p => p.MinAmount >= amount).ToList();
    }

    // Получение всех платежей
    public Dictionary<int, PaymentMethod> GetAllPayments()
    {
        return new Dictionary<int, PaymentMethod>(_paymentsDictionary);
    }

    // Обработка ожидающих платежей
    public void ProcessPendingPayments(decimal amount)
    {
        Console.WriteLine($"Обработка {_pendingPayments.Count} ожидающих платежей:");
        while (_pendingPayments.Count > 0)
        {
            var payment = _pendingPayments.Dequeue();
            payment.ProcessPayment(amount);
        }
    }

    // Группировка по категориям
    public Dictionary<string, List<PaymentMethod>> GroupByCategory()
    {
        return _paymentsList
            .GroupBy(p => p.Category)
            .ToDictionary(g => g.Key, g => g.ToList());
    }
}

// Базовый класс с событиями
public abstract class PaymentMethod
{
    public int PaymentMethodId { get; set; }
    public string MethodName { get; set; }
    public decimal MinAmount { get; set; }
    public string Currency { get; set; }
    public bool IsActive { get; set; }
    
    // Новые атрибуты
    public string Category { get; set; }
    public decimal MaxAmount { get; set; }
    public List<string> TransactionHistory { get; protected set; }
    public Dictionary<string, object> Settings { get; protected set; }

    // События
    public event PaymentProcessedEventHandler PaymentProcessed;
    public event PaymentFailedEventHandler PaymentFailed;
    public event PaymentStatusChangedEventHandler StatusChanged;

    protected PaymentMethod(int id, string name, decimal minAmount)
    {
        PaymentMethodId = id;
        MethodName = name;
        MinAmount = minAmount;
        Currency = "RUB";
        IsActive = true;
        Category = "General";
        MaxAmount = 1000000;
        TransactionHistory = new List<string>();
        Settings = new Dictionary<string, object>();
    }

    // Виртуальные методы
    public virtual void ProcessPayment(decimal amount)
    {
        if (ValidateAmount(amount))
        {
            var transactionId = Guid.NewGuid().ToString();
            TransactionHistory.Add($"Успешный платеж: {amount} {Currency} - {DateTime.Now}");
            
            // Вызов события
            OnPaymentProcessed(new PaymentEventArgs(amount, Currency, transactionId));
        }
        else
        {
            OnPaymentFailed(new PaymentEventArgs(amount, Currency, "FAILED"));
        }
    }

    public virtual bool ValidateAmount(decimal amount)
    {
        return amount >= MinAmount && amount <= MaxAmount;
    }

    // Новые методы
    public virtual Dictionary<string, decimal> GetFeeStructure()
    {
        return new Dictionary<string, decimal>
        {
            {"min_amount", MinAmount},
            {"max_amount", MaxAmount}
        };
    }

    public void AddSetting(string key, object value)
    {
        Settings[key] = value;
        Console.WriteLine($"Настройка добавлена: {key} = {value}");
    }

    public List<string> GetRecentTransactions(int count = 5)
    {
        return TransactionHistory.TakeLast(count).ToList();
    }

    // Методы для вызова событий
    protected virtual void OnPaymentProcessed(PaymentEventArgs e)
    {
        PaymentProcessed?.Invoke(this, e);
    }

    protected virtual void OnPaymentFailed(PaymentEventArgs e)
    {
        PaymentFailed?.Invoke(this, e);
    }

    protected virtual void OnStatusChanged(StatusChangedEventArgs e)
    {
        StatusChanged?.Invoke(this, e);
    }
}

// OnlinePayment с коллекциями и событиями
public class OnlinePayment : PaymentMethod
{
    public string PaymentUrl { get; set; }
    public string GatewayName { get; set; }
    
    // Новые атрибуты
    public Dictionary<string, string> GatewaySettings { get; private set; }
    public Queue<string> PendingOperations { get; private set; }
    public List<DateTime> ConnectionAttempts { get; private set; }
    public HashSet<string> SupportedCurrencies { get; private set; }

    public OnlinePayment(int id, string name, decimal minAmount, string paymentUrl, string gatewayName)
        : base(id, name, minAmount)
    {
        PaymentUrl = paymentUrl;
        GatewayName = gatewayName;
        Category = "Digital";
        MaxAmount = 500000;
        
        // Инициализация коллекций
        GatewaySettings = new Dictionary<string, string>();
        PendingOperations = new Queue<string>();
        ConnectionAttempts = new List<DateTime>();
        SupportedCurrencies = new HashSet<string> { "RUB", "USD", "EUR" };
        
        // Подписка на события
        PaymentProcessed += OnOnlinePaymentProcessed;
    }

    public override void ProcessPayment(decimal amount)
    {
        ConnectionAttempts.Add(DateTime.Now);
        
        if (ValidateAmount(amount) && SupportedCurrencies.Contains(Currency))
        {
            var transactionId = Guid.NewGuid().ToString();
            TransactionHistory.Add($"Онлайн платеж через {GatewayName}: {amount} {Currency}");
            
            Console.WriteLine($"Онлайн платеж: {amount} {Currency} через {GatewayName}");
            Console.WriteLine($"URL: {PaymentUrl}");
            
            OnPaymentProcessed(new PaymentEventArgs(amount, Currency, transactionId));
        }
        else
        {
            OnPaymentFailed(new PaymentEventArgs(amount, Currency, "VALIDATION_FAILED"));
        }
    }

    // Новые методы
    public void AddGatewaySetting(string key, string value)
    {
        GatewaySettings[key] = value;
        PendingOperations.Enqueue($"Setting updated: {key}");
        Console.WriteLine($"Настройка шлюза добавлена: {key}");
    }

    public void ProcessPendingOperations()
    {
        Console.WriteLine($"Обработка {PendingOperations.Count} операций:");
        while (PendingOperations.Count > 0)
        {
            var operation = PendingOperations.Dequeue();
            Console.WriteLine($"Обработано: {operation}");
        }
    }

    public void AddSupportedCurrency(string currency)
    {
        SupportedCurrencies.Add(currency.ToUpper());
        Console.WriteLine($"Валюта добавлена: {currency}");
    }

    public List<DateTime> GetRecentConnectionAttempts(int hours = 24)
    {
        return ConnectionAttempts
            .Where(d => d > DateTime.Now.AddHours(-hours))
            .ToList();
    }

    // Обработчик события
    private void OnOnlinePaymentProcessed(object sender, PaymentEventArgs e)
    {
        Console.WriteLine($"✅ Онлайн платеж завершен: {e.Amount} {e.Currency}");
        Console.WriteLine($"🔗 ID транзакции: {e.TransactionId}");
    }
}

// BankTransfer с коллекциями и событиями
public class BankTransfer : PaymentMethod
{
    public string BankData { get; set; }
    public decimal CommissionRate { get; set; }
    
    // Новые атрибуты
    public Dictionary<string, decimal> CommissionByAmount { get; private set; }
    public List<string> BankBranches { get; private set; }
    public Queue<DateTime> TransferSchedule { get; private set; }
    public HashSet<string> RestrictedCountries { get; private set; }

    public BankTransfer(int id, string name, decimal minAmount, string bankData, decimal commissionRate)
        : base(id, name, minAmount)
    {
        BankData = bankData;
        CommissionRate = commissionRate;
        Category = "Bank";
        MaxAmount = 2000000;
        
        // Инициализация коллекций
        CommissionByAmount = new Dictionary<string, decimal>
        {
            {"min_commission", 10m},
            {"max_commission", 5000m}
        };
        BankBranches = new List<string> { "Центральный офис" };
        TransferSchedule = new Queue<DateTime>();
        RestrictedCountries = new HashSet<string> { "North Korea", "Iran" };
        
        // Подписка на события
        PaymentProcessed += OnBankTransferProcessed;
        PaymentFailed += OnBankTransferFailed;
    }

    public override void ProcessPayment(decimal amount)
    {
        decimal commission = CalculateCommission(amount);
        decimal total = amount + commission;

        if (ValidateAmount(amount))
        {
            var transactionId = Guid.NewGuid().ToString();
            TransferSchedule.Enqueue(DateTime.Now);
            TransactionHistory.Add($"Банковский перевод: {amount} {Currency} + комиссия {commission}");
            
            Console.WriteLine($"Банковский перевод: {amount} {Currency}");
            Console.WriteLine($"Комиссия: {commission} {Currency}, Итого: {total} {Currency}");
            
            OnPaymentProcessed(new PaymentEventArgs(total, Currency, transactionId));
        }
        else
        {
            OnPaymentFailed(new PaymentEventArgs(amount, Currency, "AMOUNT_VALIDATION_FAILED"));
        }
    }

    // Новые методы
    public decimal CalculateCommission(decimal amount)
    {
        return amount * CommissionRate;
    }

    public void AddBankBranch(string branch)
    {
        BankBranches.Add(branch);
        Console.WriteLine($"Отделение банка добавлено: {branch}");
    }

    public void AddRestrictedCountry(string country)
    {
        RestrictedCountries.Add(country);
        Console.WriteLine($"Страна ограничена: {country}");
    }

    public void ProcessScheduledTransfers()
    {
        Console.WriteLine($"Запланировано переводов: {TransferSchedule.Count}");
        while (TransferSchedule.Count > 0)
        {
            var scheduleTime = TransferSchedule.Dequeue();
            Console.WriteLine($"Обработан перевод от {scheduleTime:HH:mm}");
        }
    }

    // Обработчики событий
    private void OnBankTransferProcessed(object sender, PaymentEventArgs e)
    {
        Console.WriteLine($"🏦 Банковский перевод выполнен: {e.Amount} {e.Currency}");
    }

    private void OnBankTransferFailed(object sender, PaymentEventArgs e)
    {
        Console.WriteLine($"❌ Ошибка банковского перевода: {e.Amount} {e.Currency}");
    }
}

// CashPayment с коллекциями и событиями
public class CashPayment : PaymentMethod
{
    public string CashPickupPoint { get; set; }
    
    // Новые атрибуты
    public Dictionary<string, string> PickupPoints { get; private set; }
    public List<string> Cashiers { get; private set; }
    public Queue<decimal> DailyCashFlow { get; private set; }
    public HashSet<string> AcceptedDocuments { get; private set; }

    public CashPayment(int id, string name, decimal minAmount, string cashPickupPoint)
        : base(id, name, minAmount)
    {
        CashPickupPoint = cashPickupPoint;
        Category = "Cash";
        MaxAmount = 100000;
        
        // Инициализация коллекций
        PickupPoints = new Dictionary<string, string>
        {
            { "main", "Центральная касса" },
            { "branch1", "Филиал №1" }
        };
        Cashiers = new List<string> { "Кассир 1", "Кассир 2" };
        DailyCashFlow = new Queue<decimal>();
        AcceptedDocuments = new HashSet<string> { "Паспорт", "Водительские права" };
        
        // Подписка на события
        StatusChanged += OnCashStatusChanged;
    }

    public override void ProcessPayment(decimal amount)
    {
        if (ValidateAmount(amount))
        {
            DailyCashFlow.Enqueue(amount);
            var transactionId = Guid.NewGuid().ToString();
            TransactionHistory.Add($"Наличный платеж в {CashPickupPoint}: {amount} {Currency}");
            
            Console.WriteLine($"Оплата наличными: {amount} {Currency}");
            Console.WriteLine($"Пункт: {CashPickupPoint}");
            
            OnPaymentProcessed(new PaymentEventArgs(amount, Currency, transactionId));
            OnStatusChanged(new StatusChangedEventArgs("Pending", "Completed", MethodName));
        }
        else
        {
            OnPaymentFailed(new PaymentEventArgs(amount, Currency, "CASH_LIMIT_EXCEEDED"));
        }
    }

    // Новые методы
    public void AddPickupPoint(string key, string location)
    {
        PickupPoints[key] = location;
        Console.WriteLine($"Пункт выдачи добавлен: {location}");
    }

    public void AddCashier(string cashier)
    {
        Cashiers.Add(cashier);
        Console.WriteLine($"Кассир добавлен: {cashier}");
    }

    public decimal CalculateDailyTotal()
    {
        return DailyCashFlow.Sum();
    }

    public void AddAcceptedDocument(string document)
    {
        AcceptedDocuments.Add(document);
        Console.WriteLine($"Документ принят: {document}");
    }

    // Обработчик события
    private void OnCashStatusChanged(object sender, StatusChangedEventArgs e)
    {
        Console.WriteLine($"💵 Статус наличного платежа изменен: {e.OldStatus} -> {e.NewStatus}");
    }
}

// Демонстрация
public class Program
{
    static void Main()
    {
        
    }
}
Console.WriteLine("=== КОЛЛЕКЦИИ, ДЕЛЕГАТЫ И СОБЫТИЯ ===\n");

        // Создание менеджера коллекций
        var paymentManager = new PaymentManager();

        // Создание платежных методов
        var online = new OnlinePayment(1, "Интернет-оплата", 10, "https://pay.site.com", "Stripe");
        var bank = new BankTransfer(2, "Банковский перевод", 50, "Сбербанк", 0.015m);
        var cash = new CashPayment(3, "Наличные", 1, "Касса магазина");

        // Добавление в менеджер
        paymentManager.AddPayment(online);
        paymentManager.AddPayment(bank);
        paymentManager.AddPayment(cash);

        // Демонстрация коллекций
        Console.WriteLine("1. РАБОТА С КОЛЛЕКЦИЯМИ:");
        
        online.AddGatewaySetting("timeout", "30");
        online.AddGatewaySetting("retries", "3");
        online.AddSupportedCurrency("GBP");
        
        bank.AddBankBranch("Филиал на Ленина");
        bank.AddRestrictedCountry("Syria");
        
        cash.AddPickupPoint("branch2", "ТЦ Москва");
        cash.AddCashier("Кассир 3");
        cash.AddAcceptedDocument("Заграничный паспорт");

        // Демонстрация событий
        Console.WriteLine("\n2. ДЕМОНСТРАЦИЯ СОБЫТИЙ:");
        
        // Подписка на события базового класса
        online.PaymentProcessed += (s, e) => 
            Console.WriteLine($"📧 Глобальное уведомление: Онлайн платеж {e.Amount} {e.Currency}");
        
        bank.PaymentFailed += (s, e) => 
            Console.WriteLine($"🚨 Глобальная ошибка: Банковский перевод не удался");

        // Выполнение платежей
        online.ProcessPayment(1000);
        bank.ProcessPayment(500);
        cash.ProcessPayment(5000);
        bank.ProcessPayment(5); // Должен вызвать ошибку

        // Демонстрация работы с коллекциями в менеджере
        Console.WriteLine("\n3. МЕНЕДЖЕР КОЛЛЕКЦИЙ:");
        
        var highAmountPayments = paymentManager.GetPaymentsAboveAmount(100);
        Console.WriteLine($"Платежи с мин. суммой > 100: {highAmountPayments.Count}");
        
        var grouped = paymentManager.GroupByCategory();
        foreach (var group in grouped)
        {
            Console.WriteLine($"Категория {group.Key}: {group.Value.Count} платежей");
        }

        // Демонстрация специальных методов коллекций
        Console.WriteLine("\n4. СПЕЦИАЛЬНЫЕ МЕТОДЫ КОЛЛЕКЦИЙ:");
        
        online.ProcessPendingOperations();
        bank.ProcessScheduledTransfers();
        
        var dailyTotal = cash.CalculateDailyTotal();
        Console.WriteLine($"Общая сумма наличных за день: {dailyTotal} {cash.Currency}");
        
        var recentAttempts = online.GetRecentConnectionAttempts(1);
        Console.WriteLine($"Попыток подключения за последний час: {recentAttempts.Count}");

        // Демонстрация очереди ожидающих платежей
        Console.WriteLine("\n5. ОЧЕРЕДЬ ОЖИДАЮЩИХ ПЛАТЕЖЕЙ:");
        paymentManager.ProcessPendingPayments(300);

        Console.WriteLine("\n=== ВСЕ ФУНКЦИОНАЛЬНОСТИ ДОСТУПНЫ ===");

=== КОЛЛЕКЦИИ, ДЕЛЕГАТЫ И СОБЫТИЯ ===

Платеж добавлен в менеджер: Интернет-оплата
Платеж добавлен в менеджер: Банковский перевод
Платеж добавлен в менеджер: Наличные
1. РАБОТА С КОЛЛЕКЦИЯМИ:
Настройка шлюза добавлена: timeout
Настройка шлюза добавлена: retries
Валюта добавлена: GBP
Отделение банка добавлено: Филиал на Ленина
Страна ограничена: Syria
Пункт выдачи добавлен: ТЦ Москва
Кассир добавлен: Кассир 3
Документ принят: Заграничный паспорт

2. ДЕМОНСТРАЦИЯ СОБЫТИЙ:
Онлайн платеж: 1000 RUB через Stripe
URL: https://pay.site.com
✅ Онлайн платеж завершен: 1000 RUB
🔗 ID транзакции: 190ef349-c397-43c5-804b-edd8fb464d4e
📧 Глобальное уведомление: Онлайн платеж 1000 RUB
Банковский перевод: 500 RUB
Комиссия: 7.500 RUB, Итого: 507.500 RUB
🏦 Банковский перевод выполнен: 507.500 RUB
Оплата наличными: 5000 RUB
Пункт: Касса магазина
💵 Статус наличного платежа изменен: Pending -> Completed
❌ Ошибка банковского перевода: 5 RUB
🚨 Глобальная ошибка: Банковский перевод не удался

3. МЕНЕДЖЕР КОЛЛЕКЦ